### Inspect PC variation between sparse lms, dense corresp, NSM latents
---
Do spearman's rank heatmap and PC traversal grid plot. Need to run save_pc_snapshots.py and pc_snapshot_grid.py to build NSM traversal plot, then stitch it together with lm based warp plots below.

### 1. Setup and paths

In [ ]:
# Imports and paths

import os, re, json, shutil, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import colorsys
from scipy.stats import boxcox
import plotly.graph_objects as go
import plotly.express as px
import ast
import pyvista as pv
from NSM.plotting import load_mrk_json, plot_life_history_legend, plot_family_color_legend, sort_key, generate_species_cmap_gradient, plotly_color
from NSM.morphometrics import *
import umap.umap_ as umap

# Specify training directory and atlas directory
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"  # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")

# Build other directories relative to those above
cwd      = Path.cwd()
base_wd  = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

#LM_DIR       = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"
LM_DIR       = DROPBOX_ROOT / ATLAS_RUN / "population_correspondences"
ATLAS_DIR    = DROPBOX_ROOT / ATLAS_RUN / "atlas"
#MEAN_LMS_FN  = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
MEAN_LMS_FN  = ATLAS_DIR / "atlas_dense_correspondences.mrk.json"
MEAN_MESH_FN = ATLAS_DIR / "atlas_model.ply"
atlas_mesh = pv.read(str(MEAN_MESH_FN))

OUT_DIR = Path("gmm_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Load config and parse species / vertebra from filenames (same logic as PCA_tSNE_UMAP.ipynb)
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from {config_path}\033[0m")

# Parse filenames
train_paths   = cfg["list_mesh_paths"]
all_vtk_files = [os.path.basename(f) for f in train_paths]
print(f"{len(all_vtk_files)} meshes listed in config")

pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
labels, unmatched_files = [], []
for f in all_vtk_files:
    m = pat.match(os.path.basename(f))
    if m:
        labels.append((m.group("species").strip(), m.group("vertebra").strip()))
    else:
        labels.append((None, None))
        unmatched_files.append(f)

print(f"Parsed {sum(1 for s, v in labels if s)} / {len(labels)} filenames")
if unmatched_files:
    print(f"\033[33mUnmatched ({len(unmatched_files)}), first 5:\033[0m", unmatched_files[:5])

### Load landmarks (`.mrk.json`) and build the shape array

In [ ]:
# Load 3D Slicer Atlas aligned and scaled landmark data
lm_coords = []
for fpath in all_vtk_files:
    lm_name = os.path.splitext(fpath)[0] + ".mrk.json"
    lm_path = LM_DIR / lm_name
    coords, _ = load_mrk_json(lm_path)
    lm_coords.append(coords)

lm_coords_3d = np.stack(lm_coords)   # (N, p, 3)
print(f"Landmark data shape - 3d: {lm_coords_3d.shape}")

# Atlas mean sparse landmarks (same landmark set as LM_DIR, from the setup cell)
mean_lms_3d, _ = load_mrk_json(MEAN_LMS_FN)
print(f"Atlas mean landmarks shape: {mean_lms_3d.shape}")
assert mean_lms_3d.shape == lm_coords_3d.shape[1:], (
    f"Atlas has {mean_lms_3d.shape[0]} landmarks but specimens have {lm_coords_3d.shape[1]} "
    f"-- wrong atlas run, or dense vs sparse mismatch")

In [ ]:
# Data checks before analysis

# Are the configurations already centred and scaled?
cs = np.array([centroid_size(X) for X in lm_coords_3d])
print(f"Centroid size: mean={cs.mean():.5f}  sd={cs.std():.5f}  CV={100*cs.std()/cs.mean():.2f}%  "
      f"range=({cs.min():.5f}, {cs.max():.5f})")
if 100 * cs.std() / cs.mean() > 5:
    print("\033[33mCentroid size varies by >5% — the configurations may not be fully scaled. "
          "Consider dividing each by its centroid size before proceeding.\033[0m")

consensus = mshape(lm_coords_3d)
print(f"\nConsensus vs atlas mean landmarks: Procrustes distance = "
      f"{procrustes_dist(consensus, mean_lms_3d):.6f}")
print(f"Mean per-landmark offset = {np.linalg.norm(consensus - mean_lms_3d, axis=1).mean():.6f}")

d_mean = dist_to_mean(lm_coords_3d)
print(f"\nProcrustes distance to consensus: mean={d_mean.mean():.5f}  "
      f"median={np.median(d_mean):.5f}  max={d_mean.max():.5f}")

# Optional: rescale to unit centroid size (set to True if the check above complained)
RESCALE_TO_UNIT_CS = False
if RESCALE_TO_UNIT_CS:
    lm_coords_3d = np.stack([(X - X.mean(axis=0)) / centroid_size(X) for X in lm_coords_3d])
    consensus = mshape(lm_coords_3d)
    print("\nRescaled all configurations to unit centroid size.")

### PCA of shape space (`gm.prcomp`)

In [ ]:
# Principal components of the Procrustes coordinates
pca = gm_prcomp(lm_coords_3d)
print(f"{pca['x'].shape[1]} non-trivial PCs from {pca['p']*pca['k']} coordinates")
for i in range(min(10, len(pca["prop"]))):
    print(f"  PC{i+1}: {100*pca['prop'][i]:6.2f}%   cumulative {100*pca['cum'][i]:6.2f}%")

### Spearman rank correlations

In [ ]:
# Spearman rank correlations: GMM sparse, dense landmark, and NSM latent PCs ──
# Use output tables from PCA_tSNE_UMAP_paper_figs.ipynb

def _load_pc_block(csv_name, col_prefix):
    df = pd.read_csv(OUT_DIR / csv_name)
    pc_cols = sorted(
        [c for c in df.columns if c.startswith(col_prefix)],
        key=lambda c: int(re.search(r"\d+$", c).group()))
    return df[ID_COLS + pc_cols]

def _block(prefix):
    cols = sorted([c for c in merged.columns if c.startswith(prefix)],
                  key=lambda c: int(re.search(r"\d+$", c).group()))
    return cols, merged[cols].to_numpy()

def spearman_matrix(A, B):
    R = np.empty((A.shape[1], B.shape[1]))
    P = np.empty_like(R)
    for i, j in product(range(A.shape[1]), range(B.shape[1])):
        R[i, j], P[i, j] = spearmanr(A[:, i], B[:, j])
    return R, P

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from itertools import product

# 1 ── Load score CSVs, keeping only ID + PC columns to avoid suffix conflicts ──
ID_COLS = ["specimen_id", "vertebra"]

sparse_df = _load_pc_block("../pca_tsne_umap_results/pc_sparse_points_for_stats.csv", "PC_sparse")
dense_df  = _load_pc_block("../pca_tsne_umap_results/pc_dense_points_for_stats.csv",  "PC_dense")
latent_df = _load_pc_block("../pca_tsne_umap_results/pc_latent_points_for_stats.csv", "PC_latent")

# 2 ── Merge on specimen identity (inner join → only rows present in all three) ──
merged = sparse_df.merge(dense_df, on=ID_COLS).merge(latent_df, on=ID_COLS)
n = len(merged)
print(f"{n} specimens matched across sparse / dense / latent representations")
if n < len(sparse_df):
    print(f"  ↳ {len(sparse_df) - n} specimens dropped (absent from ≥1 representation)")

sparse_cols, X_sparse = _block("PC_sparse")
dense_cols,  X_dense  = _block("PC_dense")
latent_cols, X_latent = _block("PC_latent")

# Pairs: (label_A, matrix_A, label_B, matrix_B)
PAIRS = [("LANDMARKS", X_sparse, "DENSE CORRESP",  X_dense),
        ("LANDMARKS", X_sparse, "NSM LATENTS", X_latent),
         ("DENSE CORRESP",  X_dense,  "NSM LATENTS", X_latent)]

# 5 ── Figure: one heatmap per pair ───────────────────────────────────────────

LABEL_SIZE = 35   # axis label font size
TICK_SIZE  = 30   # tick label font size
CELL_SIZE  = 30   # r-value text inside cells
CB_SIZE    = 30   # colorbar label and ticks
 
plt.rcParams.update({
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "font.size": LABEL_SIZE,
})

cmap = plt.get_cmap("PuBuGn").reversed()   # monotone: color encodes |r|, sign shown in cell text
fig, axes = plt.subplots(1, 3, figsize=(28, 8))
fig.subplots_adjust(left=0.06, right=0.88, top=0.92, bottom=0.15, wspace=0.4)
cax = fig.add_axes([0.905, 0.15, 0.018, 0.77])
 
for ax, (lbl_a, A, lbl_b, B) in zip(axes, PAIRS):
    R, P   = spearman_matrix(A, B)
    P_bonf = np.clip(P * R.size, 0, 1)   # Bonferroni over this matrix
    im = ax.imshow(abs(R), vmin=-1, vmax=1, cmap=cmap, aspect="auto")
    ax.set_yticks(range(A.shape[1]))
    ax.set_yticklabels([f"PC{i+1}" for i in range(A.shape[1])], fontsize=TICK_SIZE)
    ax.set_xticks(range(B.shape[1]))
    ax.set_xticklabels([f"PC{j+1}" for j in range(B.shape[1])],
                       rotation=45, ha="right", fontsize=TICK_SIZE)
    ax.set_ylabel(lbl_a, fontsize=LABEL_SIZE, labelpad=10)
    ax.set_xlabel(lbl_b, fontsize=LABEL_SIZE, labelpad=10)
 
    for i, j in product(range(A.shape[1]), range(B.shape[1])):
        sig  = "*" if P_bonf[i, j] < 0.05 else ""
        dark = abs(R[i, j]) > 0.55   # white text on dark cells
        ax.text(j, i,
                f"{R[i, j]:+.2f}{sig}",
                ha="center", va="center",
                fontsize=CELL_SIZE,
                color="black",
                fontweight="normal")
cb = fig.colorbar(im, cax=cax)
cb.set_label("SPEARMAN'S R", fontsize=CB_SIZE, labelpad=12)
cb.ax.tick_params(labelsize=CB_SIZE)
 
plt.savefig(OUT_DIR / "spearman_pc_correlations.png", dpi=300, bbox_inches="tight")
plt.show()

# 6 ── Print tables and write CSVs ────────────────────────────────────────────
for lbl_a, A, lbl_b, B in PAIRS:
    R, P   = spearman_matrix(A, B)
    P_bonf = np.clip(P * R.size, 0, 1)

    row_idx  = [f"{lbl_a} PC{i+1}" for i in range(A.shape[1])]
    col_idx  = [f"{lbl_b} PC{j+1}" for j in range(B.shape[1])]
    df_r = pd.DataFrame(R.round(3),  index=row_idx, columns=col_idx)
    df_p = pd.DataFrame(P_bonf.round(4), index=row_idx, columns=col_idx)

    print(f"\n{'━' * 58}")
    print(f"  {lbl_a}  ×  {lbl_b}   —   Spearman's r  (n = {n})")
    print(f"{'━' * 58}")
    print(df_r.to_string())
    print("\n  Bonferroni-corrected p-values:")
    print(df_p.to_string())

    # Per-row best match — quick read for the text
    print(f"\n  Best |r| per {lbl_a} axis:")
    for row in row_idx:
        best_col = df_r.loc[row].abs().idxmax()
        r_val    = df_r.loc[row, best_col]
        p_val    = df_p.loc[row, best_col]
        sig_flag = "  *" if p_val < 0.05 else ""
        print(f"    {row:24s}  →  {best_col:26s}  r = {r_val:+.3f}{sig_flag}")

    slug = f"{lbl_a.replace(' ', '_')}_vs_{lbl_b.replace(' ', '_')}"
    df_r.to_csv(OUT_DIR / f"spearman_r_{slug}.csv")
    df_p.to_csv(OUT_DIR / f"spearman_p_{slug}.csv")

print(f"\nAll outputs → {OUT_DIR.resolve()}")

### PC Warp Grid - use landmarks and template to warp across PC's

In [ ]:
# ── PC warp grid: sparse and dense landmarks, PC1–PC4 ────────────────────────
# Rendering matches the NSM snapshot script exactly:
#   render_cameras → crop top-right panel
#   white material, same background linspace, same z-rotation

import gc, os
import numpy as np
import cv2
import open3d as o3d
import pyvista as pv
from NSM.helper_funcs import pv_to_o3d, render_cameras

# ── Config — keep in sync with NSM snapshot script ───────────────────────────
N_PCS   = 4     # rows  (PC1 … PC4)
N_STEPS = 5     # columns per row
WIDTH   = 640   # px per panel, matches snapshot script
HEIGHT  = 480
BASE_BG_COL     = np.array([0.38, 1.0,  0.98])
MAX_TINT_BG_COL = np.array([0.03, 0.11, 0.1 ])
BG_COLORS = np.linspace(BASE_BG_COL, MAX_TINT_BG_COL, 5)  # one row per PC
ROT_MATRIX = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(13)])
#ROT_MATRIX = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, ((np.pi / 2) + np.deg2rad(13))])
RENDERERS = [o3d.visualization.rendering.OffscreenRenderer(WIDTH, HEIGHT) for _ in range(4)]

# ── Helpers ───────────────────────────────────────────────────────────────────
def _make_material():
    mat = o3d.visualization.rendering.MaterialRecord()
    mat.shader     = "defaultLit"
    mat.base_color = [1.0, 1.0, 1.0, 1.0]
    return mat

def warp_mesh(mesh, ref_lms, target_lms, mag=1.0):
    """geomorph::warpRefMesh -- deform a surface with the TPS fitted to the landmarks."""
    tgt = ref_lms + mag * (np.asarray(target_lms) - ref_lms)
    tps = tps_fit(ref_lms, tgt)
    warped = mesh.copy()
    warped.points = tps_apply(tps, np.asarray(mesh.points))
    return warped, tps

def _crop_top_right(combined):
    return combined[:HEIGHT, WIDTH:]

# Subsample dense landmarks for TPS — quality is fine at 300-500 pts,
# O(N³) makes anything above ~1000 very slow
MAX_TPS_PTS = 500

def fps(points, n):
    """Farthest point sampling — O(N*n) but n is small so fast enough."""
    selected = [np.random.randint(len(points))]
    dists    = np.full(len(points), np.inf)
    for _ in range(n - 1):
        dists    = np.minimum(dists, np.linalg.norm(points - points[selected[-1]], axis=1))
        selected.append(np.argmax(dists))
    return np.array(selected)

def _subsample_lms(ref_lms, target_lms, n=MAX_TPS_PTS):
    if len(ref_lms) <= n:
        return ref_lms, target_lms
    idx = fps(ref_lms, n)
    return ref_lms[idx], target_lms[idx]

# ── Grid builder ──────────────────────────────────────────────────────────────

def build_warp_grid(pca, mean_lms, label, out_dir):
    """Render each cell and save as individual PNG — no in-memory assembly."""
    os.makedirs(out_dir, exist_ok=True)
    mat = _make_material()
 
    for pc_idx in range(N_PCS):
        pc_dir   = os.path.join(out_dir, f"pc{pc_idx + 1}")
        os.makedirs(pc_dir, exist_ok=True)
 
        observed = pca["x"][:, pc_idx]
        scores   = np.linspace(observed.max(), observed.min(), N_STEPS)
        bg_color = BG_COLORS[pc_idx]
 
        for r in RENDERERS:
            r.scene.set_background(list(bg_color) + [1.0])
 
        for step_idx, score in enumerate(scores):
            img     = np.full((HEIGHT, WIDTH, 3),
                              (bg_color * 255).astype(np.uint8), dtype=np.uint8)
            tps_obj = None
            try:
                target             = pc_shape(pca, pc_idx, score=score)
                ref_sub, tgt_sub   = _subsample_lms(mean_lms, target)
                warped_pv, tps_obj = warp_mesh(atlas_mesh, ref_sub, tgt_sub)
                del target, tps_obj;  tps_obj = None
 
                pv_clean = warped_pv.extract_surface(algorithm='dataset_surface').triangulate()
                del warped_pv
                pv_clean = pv_clean.compute_normals(cell_normals=False, point_normals=True,
                                                    inplace=False, auto_orient_normals=True)
                o3d_mesh = pv_to_o3d(pv_clean)
                del pv_clean
                o3d_mesh.compute_vertex_normals()
                o3d_mesh.rotate(ROT_MATRIX, center=o3d_mesh.get_center())
 
                combined = render_cameras(RENDERERS, o3d_mesh, step_idx,
                                          mat, N_STEPS, n_rotations=1)
                del o3d_mesh
                img = _crop_top_right(combined)
                del combined
 
            except Exception as e:
                print(f"  Error {label} PC{pc_idx+1} step {step_idx+1}: {e}")
                import traceback; traceback.print_exc()
            finally:
                if tps_obj is not None:
                    del tps_obj
                gc.collect()
 
            fname = os.path.join(pc_dir,
                                 f"step{step_idx+1:02d}_of_{N_STEPS}.png")
            cv2.imwrite(fname, img)
            del img
            print(f"  {label}  PC{pc_idx+1}  {step_idx+1}/{N_STEPS}  score={score:.3f}  ✓",
                  flush=True)
            gc.collect()
 
    print(f"Done — PNGs saved to {out_dir}")

# ── Run ───────────────────────────────────────────────────────────────────────

# pca_sparse = pca   ← uncomment if running in the GMM notebook
print("Rendering sparse LM grid…")
build_warp_grid(pca, mean_lms_3d,
                label="dense",
                out_dir=str(OUT_DIR / "pc_snapshots_dense"))

In [ ]:
# ── Cell 2: stitch saved PNGs into a grid ────────────────────────────────────
# Run this in a separate cell after the render cell completes.

import numpy as np
import cv2
from pathlib import Path

N_PCS   = 4
N_STEPS = 5

FLIP_IDX = 0

def stitch_grid(out_dir, label):
    out_dir = Path(out_dir)
    rows = []
    for pc_idx in range(N_PCS):
        pc_dir = out_dir / f"pc{pc_idx + 1}"
        files  = sorted(pc_dir.glob("step*.png"))
        if len(files) != N_STEPS:
            print(f"  Warning: PC{pc_idx+1} has {len(files)} files, expected {N_STEPS}")
        if pc_idx == FLIP_IDX:
            imgs = [cv2.imread(str(f)) for f in reversed(files)]
        else:
            imgs = [cv2.imread(str(f)) for f in files]
        rows.append(np.hstack(imgs))

    grid      = np.vstack(rows)
    grid_path = out_dir.parent / f"pc_grid_{label}.png"
    cv2.imwrite(str(grid_path), grid)
    print(f"Grid saved → {grid_path}  ({grid.shape[1]}×{grid.shape[0]} px)")

stitch_grid(OUT_DIR / "pc_snapshots_dense", label="dense")
# stitch_grid(OUT_DIR / "pc_snapshots_sparse",  label="sparse")

In [ ]:
# Stitch traversal plots together

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

FONT_SIZE        = 15
LABEL_FONT_SIZE  = FONT_SIZE
PC_FONT_SIZE     = FONT_SIZE
DPI              = 300
LW               = 0.5

SPARSE_SIDE  = OUT_DIR / "pc_grid_sparse_side.png"
SPARSE_FRONT = OUT_DIR / "pc_grid_sparse_front.png"
DENSE_SIDE   = OUT_DIR / "pc_grid_dense_side.png"
DENSE_FRONT  = OUT_DIR / "pc_grid_dense_front.png"
NSM_SIDE     = OUT_DIR / "pc_grid_nsm_side.png"
NSM_FRONT    = OUT_DIR / "pc_grid_nsm_front.png"
OUT_FIGURE   = OUT_DIR / "pc_interpret_panel.png"

N_PCS      = 4
COL_LABELS = ["BACK", "SIDE"]
ROW_LABELS = ["LANDMARKS", "DENSE CORRESP.", "NSM LATENTS"]

GRID = [[SPARSE_FRONT, SPARSE_SIDE ],
        [DENSE_FRONT,  DENSE_SIDE  ],
        [NSM_FRONT,    NSM_SIDE    ]]

# ── Helper ────────────────────────────────────────────────────────────────────
def load_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Could not load: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ── Height ratios ─────────────────────────────────────────────────────────────
grid_ratios = [load_rgb(row[0]).shape[0] / load_rgb(row[0]).shape[1] for row in GRID]

# ── Figure ────────────────────────────────────────────────────────────────────
plt.rcParams.update({"font.size":      FONT_SIZE,
                     "font.weight":    "normal",
                     "axes.linewidth": LW,
                     "text.color":     "black",})

fig = plt.figure(figsize=(12, sum(grid_ratios) * 6), dpi=DPI)

gs = gridspec.GridSpec(3, 2,
                       figure=fig,
                        height_ratios=grid_ratios,
                        hspace=0.06,
                        wspace=0.03,
                        left=0.12, right=0.995,
                        top=0.97,  bottom=0.005)

for r, row_paths in enumerate(GRID):
    for c, fpath in enumerate(row_paths):
        img = load_rgb(fpath)
        ax  = fig.add_subplot(gs[r, c])
        ax.imshow(img, aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])

        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_linewidth(LW)
            sp.set_edgecolor("black")

        # Column headers on first row only
        if r == 0:
            ax.set_title(COL_LABELS[c], fontsize=LABEL_FONT_SIZE,
                         fontweight="normal", pad=10)

        # Row label on left column
        if c == 0:
            ax.set_ylabel(ROW_LABELS[r], fontsize=LABEL_FONT_SIZE,
                          fontweight="normal", labelpad=40)
            # PC labels
            for pc_idx in range(N_PCS):
                y = 1 - (pc_idx + 0.5) / N_PCS
                ax.text(-0.01, y, f"PC{pc_idx + 1}",
                        transform=ax.transAxes,
                        fontsize=PC_FONT_SIZE, fontweight="normal",
                        va="center", ha="right", color="black")

plt.savefig(str(OUT_FIGURE), dpi=DPI, bbox_inches="tight")
plt.close()
print(f"Saved → {OUT_FIGURE}")